<a href="https://colab.research.google.com/github/tokakhaled/AISA-ArabicFC/blob/main/unsloth_ppp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ===== Apply PP + Canon to an existing uns_2 submission  =====
import json, re, unicodedata
from datasets import load_dataset

IN_B  = "submission_trackB_2.jsonl"                 # uns_2 Track B file (superset: has think)
OUT_A = "submission_trackA_uns2_canon.jsonl"
OUT_B = "submission_trackB_uns2_canon.jsonl"
SEP   = "<start_of_turn>model"

# --- load predictions FROM THE TRACK B FILE (has think + tool_called + arguments) ---
preds = [json.loads(l) for l in open(IN_B, encoding="utf-8") if l.strip()]
preds.sort(key=lambda p: p["id"])

# --- dev set: queries (for IBAN check) + gold (for scoring) ---
val = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="dev")

def parse_model_output(text):
    out = {"function_name":"none","arguments":{},"think":""}
    if (m:=re.search(r"<think>\s*(.*?)\s*</think>",text,re.DOTALL)): out["think"]=m.group(1).strip()
    if (m:=re.search(r"<start_function_call>\s*call:(\w+)\{(.*?)\}\s*<end_function_call>",text,re.DOTALL)):
        out["function_name"]=m.group(1)
        for k,sv,nv in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))",m.group(2)):
            v=sv if sv else nv
            try: v=float(v) if "." in str(v) else int(v)
            except (ValueError,TypeError): pass
            out["arguments"][k]=v
    return out

queries = [r["text"] for r in val]
gold = [{"fn":(g:=parse_model_output(r["text"].split(SEP,1)[1]))["function_name"], "args":g["arguments"]} for r in val]

# --- norm ---
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩","0123456789")
def norm(v):
    s=str(v).translate(AR2EN).strip().lower()
    s="".join(c for c in unicodedata.normalize("NFKD",s) if not unicodedata.combining(c))
    for a,b in [("أ","ا"),("إ","ا"),("آ","ا"),("ى","ي"),("ة","ه")]: s=s.replace(a,b)
    try:
        f=float(s); s=str(int(f)) if f==int(f) else str(f)
    except ValueError: pass
    return s
def norm_args(d): return {norm(k):norm(v) for k,v in d.items()}

# --- post-processing ---
LANG_PP={}
for code,names in {"de":["german","الالمانيه","الماني"],"en":["english","الانجليزيه","انجليزي"],
 "fr":["french","الفرنسيه","فرنسي"],"es":["spanish","الاسبانيه","اسباني"],"ar":["arabic","العربيه","عربي"],
 "tr":["turkish","التركيه","تركي"],"zh":["chinese","الصينيه","صيني"],"ru":["russian","الروسيه","روسي"],
 "ur":["urdu","الارديه","اردو"],"fa":["persian","farsi","الفارسيه"],"hi":["hindi","الهنديه"],"it":["italian","الايطاليه","ايطالي"]}.items():
    for n in names: LANG_PP[norm(n)]=code
ZTYPE={norm("ذهب"):"gold",norm("فضه"):"silver",norm("مال"):"cash",norm("نقد"):"cash",norm("نقود"):"cash"}
def postprocess(fn,args,query):
    a=dict(args)
    if "target_language" in a: a["target_language"]=LANG_PP.get(norm(a["target_language"]),a["target_language"])
    if fn=="calculate_zakat" and "type" in a: a["type"]=ZTYPE.get(norm(a["type"]),a["type"])
    if "recipient_iban" in a:
        ib=str(a["recipient_iban"])
        if ib not in query and not re.search(r"[A-Z]{2}\d{2}[A-Z0-9]{8,}",query) and not re.search(r"\d{15,}",query): a.pop("recipient_iban")
    if "currency" in a and re.fullmatch(r"[a-z]{2,4}",str(a["currency"])): a.pop("currency")
    return a

# --- canonicalizer ---
ZAKAT_TYPE={"cash":["مال","المال","نقد","نقود","كاش","اموال","سيوله"],"gold":["ذهب","الذهب","ذهبي"],"silver":["فضه","الفضه","فضي"],
 "trade":["تجاره","التجاره","عروض تجاريه","بضاعه","تجاري"],"fitr":["فطر","الفطر","فطره"],"crops":["زروع","الزروع","محاصيل","زرع","حبوب"],
 "general":["عام","عامه"],"salary":["راتب","الراتب","رواتب","دخل"],"livestock":["مواشي","انعام","ماشيه","ابل","اغنام"],
 "realestate":["عقار","عقارات","العقار"],"shares":["اسهم","سهم","الاسهم"]}
TERMINATION={"resignation":["استقاله","استقال","استقلت"],"dismissal":["فصل","طرد","طردت","فصلت","مفصول","تسريح"],
 "unfair_dismissal":["فصل تعسفي","طرد تعسفي","تعسفي"],"end_of_contract":["انهاء عقد","انتهاء العقد","انتهاء عقد","نهايه العقد","انتهاء المده"],
 "retirement":["تقاعد","التقاعد","معاش","تقاعدت"],"economic":["اقتصادي","اسباب اقتصاديه","اقتصاديه"],
 "disciplinary":["تاديبي","تاديبيه","عقوبه تاديبيه"],"mutual_consent":["اتفاق","تراضي","بالتراضي","توافق"]}
LANG_SYN={"en":["انجليزي","الانجليزيه","انكليزي","english"],"fr":["فرنسي","الفرنسيه","french"],"es":["اسباني","الاسبانيه","spanish"],
 "de":["الماني","الالمانيه","german"],"it":["ايطالي","الايطاليه","italian"],"ar":["عربي","العربيه","arabic"],
 "zh":["صيني","الصينيه","chinese","مندرين"],"ja":["ياباني","اليابانيه","japanese"],"tr":["تركي","التركيه","turkish"],
 "fa":["فارسي","الفارسيه","persian","farsi"],"ko":["كوري","الكوريه","korean"]}
def _build(m):
    d={}
    for c,syns in m.items():
        d[norm(c)]=c
        for s in syns: d[norm(s)]=c
    return d
LK_Z,LK_T,LK_L=_build(ZAKAT_TYPE),_build(TERMINATION),_build(LANG_SYN)
def _canon(lk,v):
    n=norm(v)
    if n in lk: return lk[n]
    for k,c in lk.items():
        if k and k in n: return c
    return v
def canonicalize(fn,args):
    a=dict(args)
    if fn=="calculate_zakat" and "type" in a: a["type"]=_canon(LK_Z,a["type"])
    if fn=="calculate_end_of_service" and "termination_type" in a: a["termination_type"]=_canon(LK_T,a["termination_type"])
    if fn=="translate_text" and "target_language" in a: a["target_language"]=_canon(LK_L,a["target_language"])
    return a

# --- score (with ThinkRate) ---
def is_arabic(s): return any("\u0600" <= ch <= "\u06FF" for ch in s)
def score(ps):
    fn_c=arg_c=pos=think=0
    for p,g in zip(ps,gold):
        if p["tool_called"]==g["fn"]: fn_c+=1
        if g["fn"]!="none":
            pos+=1
            ga,pa=norm_args(g["args"]),norm_args(p["arguments"])
            if all(k in pa and pa[k]==v for k,v in ga.items()) and len(pa)>=len(ga): arg_c+=1
        t=p.get("think","")
        if t and is_arabic(t): think+=1
    return fn_c/len(ps), arg_c/pos, think/len(ps)

# --- apply PP + canon ---
preds_full=[]
for p in preds:
    q = queries[p["id"]]
    a = postprocess(p["tool_called"], p.get("arguments",{}), q)
    a = canonicalize(p["tool_called"], a)
    preds_full.append({**p, "arguments": a})

f0,a0,t0=score(preds)
f1,a1,t1=score(preds_full)
print(f"uns_2 as-is   : FnAcc {f0:.3f}  ArgEM {a0:.3f}  OverallA {0.4*f0+0.6*a0:.3f}  ThinkRate {t0:.3f}")
print(f"uns_2 +PP+CAN : FnAcc {f1:.3f}  ArgEM {a1:.3f}  OverallA {0.4*f1+0.6*a1:.3f}  ThinkRate {t1:.3f}")

# --- write files ---
with open(OUT_A,"w",encoding="utf-8") as f:
    for p in preds_full:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],"arguments":p["arguments"]},ensure_ascii=False)+"\n")
with open(OUT_B,"w",encoding="utf-8") as f:
    for p in preds_full:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],"arguments":p["arguments"],"think":p.get("think","")},ensure_ascii=False)+"\n")

nb=[json.loads(l) for l in open(OUT_B,encoding="utf-8")]
print(f"\nwrote {OUT_A} + {OUT_B}")
print(f"OUT_B non-empty think: {sum(1 for p in nb if p.get('think','').strip())}/{len(nb)}  (must NOT be 0)")

README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/678k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10550 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/545 [00:00<?, ? examples/s]

uns_2 as-is   : FnAcc 0.991  ArgEM 0.676  OverallA 0.802  ThinkRate 0.914
uns_2 +PP+CAN : FnAcc 0.991  ArgEM 0.676  OverallA 0.802  ThinkRate 0.914

wrote submission_trackA_uns2_canon.jsonl + submission_trackB_uns2_canon.jsonl
OUT_B non-empty think: 498/545  (must NOT be 0)
